In [ ]:
# ============================================================
# DISTÂNCIA DE MUNICÍPIOS BRASILEIROS A AEROPORTOS
# versão otimizada e geograficamente correta
# ============================================================

import pandas as pd
import numpy as np
import requests
from io import StringIO
from sklearn.neighbors import BallTree
import math
import warnings
warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Carregar sedes municipais (geobr)
# ------------------------------------------------------------
def carregar_sedes_municipais():

    from geobr import read_municipal_seat

    print("Carregando coordenadas das sedes municipais...")

    sedes = read_municipal_seat()

    df = pd.DataFrame({
        "municipio": sedes["name_muni"],
        "uf": sedes["abbrev_state"],
        "codigo_ibge": sedes["code_muni"],
        "latitude": sedes.geometry.y,
        "longitude": sedes.geometry.x
    })

    print("Total de municípios:", len(df))

    return df


# ------------------------------------------------------------
# Carregar aeroportos
# ------------------------------------------------------------
def carregar_aeroportos():

    url = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat"

    colunas = [
        "airport_id","name","city","country","iata","icao",
        "latitude","longitude","altitude","timezone",
        "dst","tz_database","type","source"
    ]

    print("Baixando base de aeroportos...")

    r = requests.get(url)

    df = pd.read_csv(
        StringIO(r.text),
        names=colunas,
        header=None,
        dtype=str,
        quotechar='"'
    )

    df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

    df = df[df["country"] == "Brazil"]
    df = df[df["type"] == "airport"]

    df = df.dropna(subset=["latitude","longitude"])

    df["iata"] = df["iata"].fillna("").str.strip().str.upper()
    df["icao"] = df["icao"].fillna("").str.strip().str.upper()

    df = df[(df["iata"]!="") | (df["icao"]!="")]

    print("Aeroportos encontrados:", len(df))

    return df.reset_index(drop=True)


# ------------------------------------------------------------
# Construir estrutura espacial
# ------------------------------------------------------------
def construir_arvore_aeroportos(df_aero):

    coords = np.radians(
        np.c_[df_aero["latitude"], df_aero["longitude"]]
    )

    tree = BallTree(coords, metric="haversine")

    return tree


# ------------------------------------------------------------
# Calcular aeroporto mais próximo
# ------------------------------------------------------------
def calcular_aeroportos_proximos(municipios, aeroportos, tree):

    print("Calculando aeroportos mais próximos...")

    coords_mun = np.radians(
        np.c_[municipios["latitude"], municipios["longitude"]]
    )

    dist, ind = tree.query(coords_mun, k=1)

    dist_km = dist.flatten() * 6371

    idx = ind.flatten()

    resultado = municipios.copy()

    resultado["aeroporto_mais_proximo"] = aeroportos.loc[idx,"name"].values
    resultado["codigo_iata"] = aeroportos.loc[idx,"iata"].values
    resultado["codigo_icao"] = aeroportos.loc[idx,"icao"].values

    resultado["codigo_principal"] = resultado["codigo_iata"]

    mask = resultado["codigo_principal"] == ""
    resultado.loc[mask,"codigo_principal"] = resultado.loc[mask,"codigo_icao"]

    resultado["distancia_km"] = dist_km.round(2)

    return resultado


# ------------------------------------------------------------
# Estatísticas
# ------------------------------------------------------------
def analisar_resultados(df):

    print("\n==============================")
    print("ESTATÍSTICAS")
    print("==============================")

    print("Municípios:",len(df))

    print("\nDistâncias (km)")

    print("Média:",df["distancia_km"].mean())
    print("Mediana:",df["distancia_km"].median())
    print("Mínima:",df["distancia_km"].min())
    print("Máxima:",df["distancia_km"].max())
    print("Desvio padrão:",df["distancia_km"].std())

    print("\nMunicípios mais distantes")

    print(
        df.nlargest(10,"distancia_km")[
            ["municipio","uf","distancia_km","codigo_principal"]
        ]
    )

    print("\nMunicípios mais próximos")

    print(
        df.nsmallest(10,"distancia_km")[
            ["municipio","uf","distancia_km","codigo_principal"]
        ]
    )

    df.to_csv(
        "aeroporto_mais_proximo_por_municipio.csv",
        index=False
    )

    print("\nArquivo salvo: aeroporto_mais_proximo_por_municipio.csv")


# ------------------------------------------------------------
# Buscar cidade específica
# ------------------------------------------------------------
def buscar_cidade(df, nome, uf):

    mask = (
        df["municipio"].str.upper().str.contains(nome.upper(),na=False)
    ) & (
        df["uf"].str.upper()==uf.upper()
    )

    r = df[mask]

    if len(r)==0:

        print("Cidade não encontrada")
        return

    print("\nResultado")

    print(
        r[[
            "municipio",
            "uf",
            "aeroporto_mais_proximo",
            "codigo_principal",
            "distancia_km"
        ]]
    )


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
def main():

    print("\n==============================")
    print("DISTÂNCIA A AEROPORTOS")
    print("==============================")

    municipios = carregar_sedes_municipais()

    aeroportos = carregar_aeroportos()

    tree = construir_arvore_aeroportos(aeroportos)

    resultados = calcular_aeroportos_proximos(
        municipios,
        aeroportos,
        tree
    )

    while True:

        print("\nMENU")
        print("1 buscar cidade")
        print("2 estatísticas gerais")
        print("3 sair")

        op = input("opção: ")

        if op=="1":

            cidade = input("cidade: ")
            uf = input("UF: ")

            buscar_cidade(resultados,cidade,uf)

        elif op=="2":

            analisar_resultados(resultados)

        elif op=="3":

            break


if __name__ == "__main__":
    main()


DISTÂNCIA A AEROPORTOS
Carregando coordenadas das sedes municipais...
Total de municípios: 5565
Baixando base de aeroportos...
Aeroportos encontrados: 264
Calculando aeroportos mais próximos...

MENU
1 buscar cidade
2 estatísticas gerais
3 sair

Resultado
     municipio  uf           aeroporto_mais_proximo codigo_principal  \
3173   Vitória  ES  Eurico de Aguiar Salles Airport              VIX   

      distancia_km  
3173          7.85  

MENU
1 buscar cidade
2 estatísticas gerais
3 sair

ESTATÍSTICAS
Municípios: 5565

Distâncias (km)
Média: 70.08647978436657
Mediana: 60.94
Mínima: 0.67
Máxima: 260.98
Desvio padrão: 46.06666721010647

Municípios mais distantes
                         municipio  uf  distancia_km codigo_principal
842                 Santa Filomena  PI        260.98              CLN
457                  Alto Parnaíba  MA        260.62              CLN
797          Monte Alegre Do Piauí  PI        260.20              BRA
683        Baixa Grande Do Ribeiro  PI        254